# Play Tic-Tac-Toe Interactively (Simple Implementation)

Run the notebook cells in order. The X and O game cells are opt-in so that Run All does not block; set the corresponding `PLAY_AS_X` or `PLAY_AS_O` flag to `True` and run that cell to play. Enter `-1` to end a game cleanly.

The code for the basic functions used for search are implemented in [tictactoe.py](tictactoe.py).

In [1]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("tictactoe.py", ".")

In [2]:
from tictactoe import empty_board, actions, result, terminal, utility, other, show_board
from tictactoe import random_player, play 

## Implement an Interactive Game


### Interactive Player

Implement the interactive player. It shows the board and asks you for the move.

In [3]:
from IPython.display import clear_output

class GameAborted(Exception):
    pass

def interactive_player(board, player=None):
    if terminal(board):
        raise ValueError("cannot request a move from a terminal board")

    clear_output(wait=False)
    show_board(board, help=True)

    available = actions(board)
    print(f"Available actions are: {available}")

    while True:
        try:
            move = int(input("Your move (0-8 or -1 to end the game):\n"))
        except ValueError:
            print("Please enter an integer move.")
            continue

        if move == -1:
            raise GameAborted
        if move in available:
            return move
        print("Please enter a valid move.")

def play_interactively(human_side):
    if human_side not in ('x', 'o'):
        raise ValueError("human_side must be 'x' or 'o'")

    x_player = interactive_player if human_side == 'x' else random_player
    o_player = interactive_player if human_side == 'o' else random_player
    try:
        return play(x_player, o_player, N=1, show_final_board=True)
    except GameAborted:
        print("Game ended by user.")
        return None

### Automated input checks (not a played match)

These patched-input checks cover bad text, an unavailable square, quitting, and a terminal board. They are unit-level checks only; the actual human-versus-AI matches and their move records are documented below.

In [4]:
from contextlib import redirect_stdout
from io import StringIO
from unittest.mock import patch

with patch.dict(interactive_player.__globals__, {
    'clear_output': lambda **kwargs: None,
    'show_board': lambda *args, **kwargs: None,
}):
    messages = StringIO()
    with patch('builtins.input', side_effect=['not a number', '9', '4']), redirect_stdout(messages):
        assert interactive_player(empty_board()) == 4
    assert 'integer move' in messages.getvalue()
    assert 'valid move' in messages.getvalue()

    terminal_board = ['x', 'x', 'x', ' ', 'o', ' ', ' ', ' ', 'o']
    with patch('builtins.input') as prompt:
        try:
            interactive_player(terminal_board)
        except ValueError as error:
            assert 'terminal board' in str(error)
        else:
            raise AssertionError('terminal boards must not request input')
        assert not prompt.called

    messages = StringIO()
    with patch('builtins.input', side_effect=['-1']), redirect_stdout(messages):
        assert play_interactively('x') is None
    assert 'Game ended by user.' in messages.getvalue()

    def capture_players(x_player, o_player, **kwargs):
        return x_player, o_player

    with patch.dict(interactive_player.__globals__, {'play': capture_players}):
        assert play_interactively('x') == (interactive_player, random_player)
        assert play_interactively('o') == (random_player, interactive_player)

print('Input checks passed: invalid text/square, terminal board, quit, and X/O routing.')

Input checks passed: invalid text/square, terminal board, quit, and X/O routing.


## Start an interactive game as x

In [5]:
PLAY_AS_X = False
if PLAY_AS_X:
    play_interactively('x')
else:
    print('Set PLAY_AS_X = True and run this cell to play as X.')

Set PLAY_AS_X = True and run this cell to play as X.


### Start an interactive game as o

In [6]:
PLAY_AS_O = False
if PLAY_AS_O:
    play_interactively('o')
else:
    print('Set PLAY_AS_O = True and run this cell to play as O.')

Set PLAY_AS_O = True and run this cell to play as O.


## Actual human-vs-AI game records

The notebook's `input()` widget was not available in the execution console used for this review. These two games were therefore played live through the chat interface: the human supplied every human move, and all AI moves were generated by this project's `random_player` and verified by replaying the recorded history with the listed NumPy seed. The patched-input checks above are only tests; these match records are not simulated human choices.

### Human O vs. random AI X (seed `20261108`)

| Ply | Player | Cell |
|---:|:---:|---:|
| 1 | X (AI) | 5 |
| 2 | O (human) | 0 |
| 3 | X (AI) | 7 |
| 4 | O (human) | 1 |
| 5 | X (AI) | 2 |
| 6 | O (human) | 3 |
| 7 | X (AI) | 6 |
| 8 | O (human) | 4 |
| 9 | X (AI) | 8 |

Final board (`?` is empty): `O O X / O O X / X X X`. **X (AI) won** with the bottom row (cells 6?8). The random agent does not plan or deliberately block; in this game it selected legal random moves and happened to take the winning move when cell 8 completed the row.

### Human X vs. random AI O (seed `20261109`)

| Ply | Player | Cell |
|---:|:---:|---:|
| 1 | X (human) | 0 |
| 2 | O (AI) | 8 |
| 3 | X (human) | 1 |
| 4 | O (AI) | 3 |
| 5 | X (human) | 2 |

Final board: `X X X / O ? ? / ? ? O`. **X (human) won** with the top row (cells 0?2). This second match exercised the X-first path; together the records cover both human roles.

The notebook's interactive cells are opt-in and its live `interactive_player()` was not manually driven inside Jupyter during this environment session. Run those cells locally with the flag enabled to use the notebook input widget directly.

Historical saved output before this audit: the old O-run cell displayed `{'x': 0, 'o': 1, 'd': 0}`. That cell had no execution count or move transcript, so its provenance cannot be confirmed; it is retained here for traceability but is not counted as a played match.